# Data preparation
Builds **Partition B** of the SemEval-2020 Task 9 SentiMix Hinglish corpus and the shared **Dakshina** normalization dictionary used by all three tasks.

- **Inputs:** downloaded automatically (SemEval files from GitHub, Dakshina v1.0 from Google)
- **Outputs:** `data/partition_b_{train,validation,test}.json`, `data/dakshina_norm_dict.json`
- **Runtime:** CPU, about 2 minutes. Run this notebook first.

## Setup

In [ ]:
# Work in a folder on Google Drive so all notebooks share data/ and results/
import os
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/NLP_Group17"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

# The Dakshina archive is large and only needed once, so keep it on Colab's local disk
os.environ["DAKSHINA_TAR"] = "/content/dakshina_dataset_v1.0.tar"

## Configuration
Seed, data folder and download sources.

In [ ]:
import os
import re
import json
import random
import urllib.request
import tarfile
from collections import defaultdict

SEED = 42
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

SEMEVAL_URLS = {
    "Train.txt": "https://raw.githubusercontent.com/singhnivedita/SemEval2020-Task9/master/Pre%20Processing%20Code/Original%20Dataset/Train.txt",
    "Validation.txt": "https://raw.githubusercontent.com/singhnivedita/SemEval2020-Task9/master/Pre%20Processing%20Code/Original%20Dataset/Validation.txt",
    "Test.txt": "https://raw.githubusercontent.com/singhnivedita/SemEval2020-Task9/master/Pre%20Processing%20Code/Original%20Dataset/Test.txt",
    "Ty.txt": "https://raw.githubusercontent.com/singhnivedita/SemEval2020-Task9/master/Fully%20Processed%20Datasets/Ty.txt"
}

DAKSHINA_URL = "https://storage.googleapis.com/gresearch/dakshina/dakshina_dataset_v1.0.tar"

## Downloading the data

In [ ]:
def download_files():
    for fname, url in SEMEVAL_URLS.items(): 
        fpath = os.path.join(DATA_DIR, fname)
        if not os.path.exists(fpath):
            print(f"Downloading {fname}...")
            urllib.request.urlretrieve(url, fpath)

## Parsing the SemEval files
Each tweet is a header line `meta<TAB>id<TAB>sentiment` followed by one `token<TAB>language-tag` line per token. Test-set labels come from `Ty.txt`. A line counts as a header only if it matches that pattern exactly, so tokens such as *metaphysist* never start a new tweet.

In [ ]:
def load_test_labels(ty_path):
    test_labels = {}
    with open(ty_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.lower().startswith("uid"):
                continue
            parts = [p.strip() for p in re.split(r"[,\t]", line) if p.strip()]
            if len(parts) >= 2:
                test_labels[parts[0]] = parts[1].lower()
    return test_labels

In [ ]:
def is_header_line(line):
    parts = line.split("\t")
    if parts[0] != "meta":
        return False, None, None
    if len(parts) >= 3 and parts[1].strip().isdigit() and parts[2].strip().lower() in {"positive", "negative", "neutral"}:
        return True, parts[1].strip(), parts[2].strip().lower()
    if len(parts) == 2 and parts[1].strip().isdigit():
        return True, parts[1].strip(), None
    # Handle cases where space/tab formatting varies slightly in header
    tokens = line.split()
    if len(tokens) >= 2 and tokens[0] == "meta" and tokens[1].isdigit():
        sent = tokens[2].lower() if len(tokens) >= 3 and tokens[2].lower() in {"positive", "negative", "neutral"} else None
        return True, tokens[1], sent
    return False, None, None

## Cleaning URLs and mentions
Links are split across tokens (`https // t . co / <id>`), so every piece is removed, including the short-link id and truncated links. `@` and the username after it are removed. Hashtags and emojis are kept as sentiment cues.

In [ ]:
URL_PIECES = {"//", ":", "://", ".", "t", "co", "c", "t.co"}  # "/" handled separately (it precedes the id)
SHORTLINK_ID = re.compile(r"^[A-Za-z0-9]{6,}$")


def _is_ellipsis(tok):
    return "…" in tok or "â€¦" in tok or tok == "..."

In [ ]:
def clean_tokens(tokens, lang_tags):
    """
    Documented consistent handling of split URLs and usernames (Requirement 6):

    * URLs: in the corpus a link is split into tokens like
      ``https // t . co / oxf8tr3bly`` (sometimes truncated, e.g. ``https // t . c …``).
      When a token contains ``http``, everything from ``http`` onwards in that token is
      dropped (any word glued in front, e.g. ``Yadavhttps``, is kept), then every
      following URL fragment is dropped, including the short-link id that follows
      ``co /``. A 10-character id left at the very end of a tweet right after
      an ellipsis (a mangled, truncated link) is also dropped.
    * Mentions: ``@`` is its own token (or glued to preceding punctuation, e.g.
      ``.@``) and the username is the next token. The ``@`` and the username are
      dropped (leading punctuation is kept), as are tokens that start with ``@``.
    """
    cleaned_toks, cleaned_tags = [], []
    i, n = 0, len(tokens)
    while i < n:
        tok, tag = tokens[i], lang_tags[i]
        low = tok.lower()

        pos = low.find("http")
        if pos != -1:
            if pos > 0:  # keep a word glued in front of the URL
                cleaned_toks.append(tok[:pos])
                cleaned_tags.append(tag)
            i += 1
            while i < n:
                frag = tokens[i]
                if frag in URL_PIECES or _is_ellipsis(frag):
                    i += 1
                    continue
                if frag.startswith("/"):
                    i += 1
                    # a plain "/" is followed by the short-link id; "/…" means the URL was truncated
                    if frag == "/" and i < n and SHORTLINK_ID.match(tokens[i]):
                        i += 1
                break
            continue

        if tok.endswith("@"):
            if tok[:-1]:  # e.g. ".@" -> keep "."
                cleaned_toks.append(tok[:-1])
                cleaned_tags.append(tag)
            i += 2  # drop "@" and the username after it
            continue
        if tok.startswith("@"):
            i += 1
            continue

        if (i == n - 1 and i > 0 and _is_ellipsis(tokens[i - 1])
                and re.fullmatch(r"[A-Za-z0-9]{10}", tok) and re.search(r"\d", tok)):
            i += 1  # trailing id of a truncated link
            continue

        cleaned_toks.append(tok)
        cleaned_tags.append(tag)
        i += 1

    if not cleaned_toks:
        return tokens, lang_tags
    return cleaned_toks, cleaned_tags

## Reading a file into tweets
Each tweet keeps its cleaned token list, the aligned language tags and the space-joined text.

In [ ]:
def parse_semeval_file(filepath, test_labels=None):
    tweets = []
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.read().splitlines()

    curr_id, curr_sent, curr_tokens, curr_tags = None, None, [], []

    def flush():
        if curr_id is not None and curr_tokens:
            sent = curr_sent if curr_sent else (test_labels.get(curr_id) if test_labels else None)
            if sent in {"positive", "negative", "neutral"}:
                c_toks, c_tags = clean_tokens(curr_tokens, curr_tags)
                tweets.append({
                    "id": curr_id,
                    "sentiment": sent,
                    "tokens": c_toks,
                    "lang_tags": c_tags,
                    "raw_text": " ".join(c_toks)
                })

    for raw_line in lines:
        line = raw_line.strip()
        if not line:
            flush()
            curr_id, curr_sent, curr_tokens, curr_tags = None, None, [], []
            continue
        is_hdr, tid, tsent = is_header_line(line)
        if is_hdr:
            flush()
            curr_id, curr_sent, curr_tokens, curr_tags = tid, tsent, [], []
        else:
            parts = line.split("\t")
            if len(parts) >= 2:
                tok, ltag = parts[0].strip(), parts[-1].strip()
            else:
                sp = line.split()
                tok = sp[0] if sp else ""
                ltag = sp[-1] if len(sp) > 1 else "O"
            if tok:
                # Normalize lang tag to {Hin, Eng, O}
                if ltag.lower().startswith("hin"):
                    ltag = "Hin"
                elif ltag.lower().startswith("eng"):
                    ltag = "Eng"
                else:
                    ltag = "O"
                curr_tokens.append(tok)
                curr_tags.append(ltag)
    flush()
    return tweets

## Dakshina normalization dictionary
From the Hindi word-pair lexicon (train, dev and test combined), every attested romanization of a Devanagari word is mapped to that word's most frequent romanization.

In [ ]:
def build_dakshina_dictionary():
    dict_path = os.path.join(DATA_DIR, "dakshina_norm_dict.json")
    if os.path.exists(dict_path):
        print("Loaded cached Dakshina dictionary.")
        with open(dict_path, "r", encoding="utf-8") as f:
            return json.load(f)

    # DAKSHINA_TAR lets Colab keep this large archive on local disk instead of Drive
    tar_path = os.environ.get("DAKSHINA_TAR", os.path.join(DATA_DIR, "dakshina_dataset_v1.0.tar"))
    if not os.path.exists(tar_path):
        print("Downloading Dakshina dataset (for Hindi lexicon)...")
        urllib.request.urlretrieve(DAKSHINA_URL, tar_path)

    lexicon_files = [
        "dakshina_dataset_v1.0/hi/lexicons/hi.translit.sampled.train.tsv",
        "dakshina_dataset_v1.0/hi/lexicons/hi.translit.sampled.dev.tsv",
        "dakshina_dataset_v1.0/hi/lexicons/hi.translit.sampled.test.tsv"
    ]
    native_to_rom = defaultdict(lambda: defaultdict(int))

    print("Extracting Dakshina Hindi transliteration lexicon...")
    with tarfile.open(tar_path, "r") as tar:
        for member_name in lexicon_files:
            f = tar.extractfile(member_name)
            if f:
                for line in f.read().decode("utf-8").splitlines():
                    parts = line.strip().split("\t")
                    if len(parts) >= 3:
                        native, rom, count = parts[0].strip(), parts[1].strip().lower(), int(parts[2].strip())
                        native_to_rom[native][rom] += count

    # For each native Devanagari word, map all attested romanizations to the highest-frequency canonical romanization
    variant_to_canonical = {}
    for native, rom_counts in native_to_rom.items():
        canonical = max(rom_counts.items(), key=lambda x: (x[1], x[0]))[0]
        for rom in rom_counts:
            variant_to_canonical[rom] = canonical

    with open(dict_path, "w", encoding="utf-8") as f:
        json.dump(variant_to_canonical, f, ensure_ascii=False)
    print(f"Built Dakshina normalization dictionary with {len(variant_to_canonical)} Romanized Hindi mappings.")
    return variant_to_canonical

## Building Partition B
Merges the three files, keeps one copy of the ~2,650 ids duplicated between train and validation, shuffles the pool with seed 42, takes the middle third (Partition B), and splits it 80/10/10 with a second seed-42 shuffle.

In [ ]:
def main():
    download_files()
    test_labels = load_test_labels(os.path.join(DATA_DIR, "Ty.txt"))

    train_tweets = parse_semeval_file(os.path.join(DATA_DIR, "Train.txt"))
    val_tweets = parse_semeval_file(os.path.join(DATA_DIR, "Validation.txt"))
    test_tweets = parse_semeval_file(os.path.join(DATA_DIR, "Test.txt"), test_labels=test_labels)

    # Deduplicate by tweet ID preserving deterministic order
    dedup_pool = {}
    for t in train_tweets + val_tweets + test_tweets:
        if t["id"] not in dedup_pool:
            dedup_pool[t["id"]] = t

    pool_list = list(dedup_pool.values())
    print(f"Total deduplicated labelled tweets: {len(pool_list)}")

    # Shuffle once with fixed seed 42 and cut into three equal thirds (Partitions A, B, C)
    rng = random.Random(SEED)
    rng.shuffle(pool_list)

    n = len(pool_list)
    third = n // 3
    partition_b = pool_list[third : 2 * third]
    print(f"Partition B total tweets: {len(partition_b)}")

    # Second shuffle within Partition B with fixed seed 42, split 80/10/10
    rng_b = random.Random(SEED)
    rng_b.shuffle(partition_b)

    nb = len(partition_b)
    n_train = int(0.8 * nb)
    n_val = int(0.1 * nb)

    splits = {
        "train": partition_b[:n_train],
        "validation": partition_b[n_train : n_train + n_val],
        "test": partition_b[n_train + n_val :]
    }

    for split_name, data in splits.items():
        out_path = os.path.join(DATA_DIR, f"partition_b_{split_name}.json")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"Saved Partition B {split_name}: {len(data)} tweets -> {out_path}")

    build_dakshina_dictionary()

## Run

In [ ]:
main()

The splits and the dictionary are now in `data/`. Next: `task1/task1_baseline.ipynb`.